# Encoder / BERT Exercises

These exercises cover:
- Tokenization & special tokens
- Encoder embeddings and attention
- Pre-training objectives: MLM and NSP
- Fine-tuning example

In [1]:
import torch
from transformers import (
    BertTokenizer,
    BertModel,
    BertForMaskedLM,
    BertForNextSentencePrediction,
    BertForSequenceClassification,
)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

## Exercise 1 — Tokenization (white-space vs WordPiece)

1) Tokenize the sentence with a white-space tokenizer.
2) Tokenize the same sentence with BERT WordPiece.
3) Explain (1–2 sentences) why WordPiece is helpful for out-of-vocabulary words.


In [19]:
text = "BERT preprocessing is essential."

def white_space_tokenize(text):
  text = text.strip()
  if not text:
      return []
  tokens = text.split()
  return tokens

# TODO 1: white-space tokenization
white_tokens = text.split(" ")
# TODO 2: BERT WordPiece tokens
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

wp_tokens = tokenizer.tokenize(text)

print('white-space:', white_tokens)
print('wordpiece:', wp_tokens)


white-space: ['BERT', 'preprocessing', 'is', 'essential.']
wordpiece: ['bert', 'prep', '##ro', '##ces', '##sing', 'is', 'essential', '.']


## Exercise 2 — Special Tokens ([CLS], [SEP])

BERT uses special tokens to mark the beginning and separation of sequences.

1) Tokenize a sentence and print the tokens.
2) Print the token IDs.
3) Identify which tokens are [CLS] and [SEP].


In [20]:
text = "Transformers are powerful models."

# TODO 1: Tokenize and get tokens
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

tokens = tokenizer.tokenize(text)

# TODO 2: Get token IDs

token_ids = tokenizer.convert_tokens_to_ids(tokens)

print('tokens:', tokens)
print('token_ids:', token_ids)
print('[CLS] id:', tokenizer.cls_token_id)
print('[SEP] id:', tokenizer.sep_token_id)


tokens: ['transformers', 'are', 'powerful', 'models', '.']
token_ids: [19081, 2024, 3928, 4275, 1012]
[CLS] id: 101
[SEP] id: 102


## Exercise 3 — Attention Outputs and Shapes

1) Run BERT with `output_attentions=True`.
2) Print:
   - number of layers
   - attention tensor shape for the first layer

Recall attention weights shape: `(batch, heads, seq_len, seq_len)`.


In [23]:
text = "BERT's attention mechanism is fascinating."

# TODO: load model and run with output_attentions=True
model = BertModel.from_pretrained('bert-base-uncased')
inputs = tokenizer(text, return_tensors='pt')
outputs = model(**inputs, output_attentions=True)

att = outputs.attentions

print('num layers:', len(att))
print('layer0 shape:', att[0].shape)  #(batch=1, heads=12, seq_len, seq_len)
print('Expected: (1, 12, seq_len, seq_len) for BERT-base with 12 attention heads')

BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


num layers: 12
layer0 shape: torch.Size([1, 12, 10, 10])
Expected: (1, 12, seq_len, seq_len) for BERT-base with 12 attention heads


## Exercise 4 — Extracting the [CLS] Token Embedding

The [CLS] token's final hidden state is often used as a sentence representation.

1) Run BERT on a sentence.
2) Extract the [CLS] token embedding (first token) from the last hidden state.
3) Print its shape.


In [26]:
text = "BERT produces contextualized embeddings."

# TODO 1: Load model and tokenize
model = BertModel.from_pretrained('bert-base-uncased')
inputs = tokenizer(text, return_tensors='pt')

# TODO 2: Run forward pass
outputs = model(**inputs, output_attentions=True)

cls_embedding = outputs.last_hidden_state[:, 0, :] #(batcjh_size, hidden_size)

print('CLS embedding shape:', cls_embedding.shape)

CLS embedding shape: torch.Size([1, 768])


## Exercise 5 — Masked Language Modeling (Simple Prediction)

Use `BertForMaskedLM` to predict a masked word.

1) Create a sentence with a [MASK] token.
2) Use the model to predict the masked token.
3) Print the top 3 predictions.


In [34]:
text = "The capital of France is [MASK]."

#load MLM model and tokenize
mlm_model = BertForMaskedLM.from_pretrained('bert-base-uncased')
inputs = tokenizer(text, return_tensors='pt')

with torch.no_grad():
    outputs = mlm_model(**inputs, output_attentions=True) # TODO 2: Get model outputs

# TODO 3: Get logits for the masked token
mask_token_index = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
logits = outputs.logits[0, mask_token_index, :]
top_k = torch.topk(logits, k = 3, dim=1)

print('Top 3 predictions:')
for i, (token_id, score) in enumerate(zip(top_k.indices[0], top_k.values[0])):
    predicted_token = tokenizer.decode([token_id])
    print(f'{i+1}. {predicted_token} (score: {score.item():.2f})')


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Top 3 predictions:
1. paris (score: 12.35)
2. lille (score: 10.58)
3. lyon (score: 10.46)


## Exercise 6 — Next Sentence Prediction (NSP)

BERT was pre-trained to predict if sentence B follows sentence A.

1) Create a pair of sentences that are related.
2) Create a pair of sentences that are unrelated.
3) Use `BertForNextSentencePrediction` to classify both pairs.
4) Print the predictions (0 = next sentence, 1 = not next sentence).


In [39]:
#related sentences
sent_a1 = "I love pizza."
sent_b1 = "It's my favorite food."

#Unrelated sentences
sent_a2 = "I love pizza."
sent_b2 = "The weather is nice today."

nsp_model = BertForNextSentencePrediction.from_pretrained('bert-base-uncased')

# TODO 2: Tokenize both pairs
inputs1 = nsp_model(sent_a1,sent_b1,return_tensors='pt')
inputs2 = nsp_model(sent_a2,sent_b2,return_tensors='pt')

# TODO 3: Get predictions
outputs1 = nsp_model(**inputs1, next_sentence_label=torch.LongTensor([1]))
outputs2 = nsp_model(**inputs2, next_sentence_label=torch.LongTensor([1]))


pred1 = outputs1
pred2 = outputs2

print('Related sentences prediction:', pred1, '(0=next, 1=not next)')
print('Unrelated sentences prediction:', pred2, '(0=next, 1=not next)')
print('\nNote: The model should predict 0 for related and 1 for unrelated,')
print('but results may vary as NSP is a weak signal.')


AttributeError: 'str' object has no attribute 'size'

## Exercise 7 — Fine-tuning Head (Sequence Classification)

Run a forward pass with `BertForSequenceClassification` and inspect:
- logits shape
- predicted label (argmax)

*(Note: the base checkpoint is not fine-tuned for sentiment, so the prediction is not meaningful; this is about understanding the architecture.)*


In [ ]:
text = "This movie was amazing!"

# TODO: Load model and tokenize
model = None
inputs = None

# TODO: Forward pass and prediction
outputs = None
pred = None

print('logits shape:', outputs.logits.shape)  #(batch_size, num_labels)
print('pred:', pred)
print('\nNote: This is a randomly initialized classification head,')
print('so the prediction is meaningless. In practice, you would fine-tune')
print('this model on a labeled dataset (e.g., sentiment analysis).')


## Exercise 8 — Visualizing Attention Patterns

Attention weights show which tokens the model focuses on when processing each token.

1) Install bertviz: `pip install bertviz`
2) Run BERT and extract attention weights.
3) Visualize the attention pattern for a specific layer and head.
4) Observe: Which tokens does [CLS] attend to? Do you see any patterns?

**Bonus**: Try different sentences and see how attention changes!

In [ ]:
# Install bertviz if needed
# !pip install bertviz

from bertviz import head_view

text = "The cat sat on the mat."

# TODO 1: Load model and tokenize
model = None
inputs = None

# TODO 2: Run model with output_attentions=True
outputs = None

# TODO 3: Extract tokens and attention
tokens = None
attention = None

tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
attention = outputs.attentions  # Tuple of (batch, heads, seq, seq) tensors

# TODO 4: Visualize attention for all heads in layer 0
# Hint: Use head_view(attention, tokens, layer=0)